# UrbanVerse-100K — Usage Demo

This notebook demonstrates the basic usage of the `urbanverse_asset` package for exploring the large-scale UrbanVerse-100K asset database, including dataset exploration, object asset retrieval, material downloads, and interactive browser-based visualization.


**Dataset:** [Oatmealliu/UrbanVerse-100K](https://huggingface.co/datasets/Oatmealliu/UrbanVerse-100K)

**API Cheatsheet:** [urbanverse_asset API cheatsheet](https://github.com/VAIL-UCLA/UrbanVerse/tree/main/urbanverse_100k)

**Package:** [urbanverse-asset source code](https://github.com/VAIL-UCLA/UrbanVerse/tree/main/urbanverse_100k)


## Table of Contents

1. [Installation & Authentication](#1.-Installation-&-Authentication)
2. [Import & Configuration](#2.-Import-&-Configuration)
3. [Dataset Metadata](#3.-Dataset-Metadata)
4. [Exploring Object Assets](#4.-Exploring-Object-Assets)
   - 4.1 [Get All UIDs](#4.1-Get-All-UIDs)
   - 4.2 [Browse the Category Hierarchy](#4.2-Browse-the-Category-Hierarchy)
   - 4.3 [Visualize Category Distribution](#4.3-Visualize-Category-Distribution)
   - 4.4 [Understand Per-Asset Annotations](#4.4-Understand-Per-Asset-Annotations)
5. [Filtering & Searching Objects](#5.-Filtering-&-Searching-Objects)
   - 5.1 [Semantic Text Search](#5.1-Semantic-Text-Search)
   - 5.2 [Combined Attribute + Text Search](#5.2-Combined-Attribute-+-Text-Search)
6. [Downloading & Viewing Object Assets](#6.-Downloading-&-Viewing-Object-Assets)
   - 6.1 [Interactive 3D Viewer — Vehicles](#6.1-Interactive-3D-Viewer-—-Vehicles)
   - 6.2 [Interactive 3D Viewer — Barriers](#6.2-Interactive-3D-Viewer-—-Barriers)
7. [Material Assets](#7.-Material-Assets)
   - 7.1 [Sky Materials](#7.1-Sky-Materials)
   - 7.2 [Road Materials](#7.2-Road-Materials)
   - 7.3 [Sidewalk Materials](#7.3-Sidewalk-Materials)
   - 7.4 [Terrain Materials](#7.4-Terrain-Materials)
8. [Vegetation Assets](#8.-Vegetation-Assets)
9. [Download the Entire Dataset](#9.-Download-the-Entire-Dataset-(Optional))
10. [Integrity Check & Repair](#10.-Integrity-Check-&-Repair)

## 1. Installation & Authentication

Install the package from PyPI (skip if already installed):

In [ ]:
!pip install urbanverse-asset

> **Request Access to the UrbanVerse-100K Dataset (if not already done):** First, request access to the UrbanVerse-100K repository by visiting the repository page [here](https://huggingface.co/datasets/Oatmealliu/UrbanVerse-100K). Fill out the access request form and click **“Submit.”** Once your request is approved, you will be able to use the toolkit to download and access the dataset files.


Authenticate your machine with the Hugging Face account that has access to the repository approved:

In [ ]:
!hf auth login

Note: You might be prompted to paste your Hugging Face access token if you are not logged in. You can create a token [here](https://huggingface.co/settings/tokens).

After login, verify the current account:

!hf auth whoami

## 2. Import & Configuration

In [ ]:
import urbanverse_asset as uva

By default, all downloaded assets are cached in `~/.cache/urbanverse/`.
You can override this with `uva.set()` — the choice persists across Python sessions.

In [ ]:
# (Optional) set a custom cache directory
# Default: ~/.cache/urbanverse/
uva.set("~/datasets_cache/urbanverse")

## 3. Dataset Metadata

Fetch high-level statistics about the UrbanVerse-100K dataset.
The master annotation is downloaded on the first call and cached in memory afterwards.

In [ ]:
meta = uva.info()

print("Number of assets:", meta["statistics"]["number_of_assets"])
print("L1 classes:      ", meta["statistics"]["number_of_classes_l1"])
print("L2 classes:      ", meta["statistics"]["number_of_classes_l2"])
print("L3 classes:      ", meta["statistics"]["number_of_classes_l3"])
print("License:         ", meta["license"])

## 4. Exploring Object Assets

### 4.1 Get All UIDs

Each 3D object asset in the dataset has a unique identifier (UID).

In [ ]:
all_uids = uva.object.get_uids_all()
print(f"Total UIDs: {len(all_uids)}")
print(f"First 5:    {all_uids[:5]}")

### 4.2 Browse the Category Hierarchy

The dataset organizes assets into a 3-level category hierarchy (L1 → L2 → L3).

In [ ]:
l1_cats = uva.object.categories(level=1) # level: 1 (top), 2, 3 (leaf)
print("L1 categories:", list(l1_cats.keys()))
for cat, cat_uids in l1_cats.items():
    print(f"  {cat}: {len(cat_uids)} assets")

### 4.3 Visualize Category Distribution

Open an interactive sunburst chart of the full category hierarchy in your browser.

In [ ]:
uva.viewer.object_distribution()

### 4.4 Understand Per-Asset Annotations

Each asset has a rich annotation (geometry, materials, affordances, physics, etc.).
Use `explain_annotation()` to learn what each field means.

In [ ]:
uva.object.explain_annotation()

In [ ]:
uva.object.explain_annotation("mass")

## 5. Filtering & Searching Objects

`get_uids_conditioned()` lets you filter assets by **attribute constraints**
(categories, dimensions, physics properties, etc.) and/or a **natural-language text query**. This is probably one of the most useful and frequently used functions for selecting assets from the dataset.

*Each filtering condition (function parameter) can be used either individually or in combination. When multiple conditions are provided, they are applied using **AND** logic.*

> **Note:** We understand that many users may not wish/need to download the full TB-scale dataset. Therefore, we provide a flexible asset search interface that supports multiple attribute filters and free-form text queries. This allows users to discover relevant assets **without downloading any disk-heavy files** (e.g., `.glb`, `.png`).

> **Note:** Free-form text queries are matched to asset annotations using a lightweight natural-language retrieval mechanism based on **SentenceBERT**. Of course, this interface can be extended to support img-img/txt-img queries (e.g., via DINOv3 or CLIP), though doing so would require downloading the associated image files. Here we pursue a lightweight design that avoids such downloads.


### 5.1 Semantic Text Search

Find vehicles matching a natural-language description using a sentence embedding model.

In [ ]:
vehicle_uids = uva.object.get_uids_conditioned(
    categories=["vehicle"],
    query="Old yellow Italian style two-door car",
    embedding_model="sentence-transformers/all-mpnet-base-v2",
    top_k=10,
)
print(f"Found {len(vehicle_uids)} matching vehicles")

### 5.2 Combined Attribute + Text Search

Mix categorical, dimensional, and semantic filters to find specific assets.

In [ ]:
barrier_uids = uva.object.get_uids_conditioned(
    categories=["traffic cone", "traffic barricade", "jersey barrier"],
    height_range=(0.0, 1.2),
    query="Yellow obstacle",
    top_k=5,
)
print(f"Found {len(barrier_uids)} matching barriers")

## 6. Downloading & Viewing Object Assets

Download GLB models, annotations, thumbnails, and renders for the selected UIDs.
Already-cached files are skipped automatically.

In [ ]:
# Download asset combos (annotation/.glb/thumbnail/renders) of the given UIDs. 
# Also, you can control what to download via the what parameter: what=("std_glb", "std_annotation", "thumbnail", "render")
result = uva.object.load(vehicle_uids)

first_uid = vehicle_uids[0]
print(f"UID:        {first_uid}")
print(f"GLB path:   {result[first_uid]['std_glb']}")
print(f"Annotation: {result[first_uid]['std_annotation']}")
print(f"Thumbnail:  {result[first_uid]['thumbnail']}")

### 6.1 Interactive 3D Viewer — Vehicles

Opens a browser-based viewer with the 3D GLB model, thumbnail, renders, and annotation table.
Use arrow keys or A/D to navigate between assets. Mouse drag to orbit, scroll to zoom.

In [ ]:
uva.viewer.object_show(vehicle_uids)

### 6.2 Interactive 3D Viewer — Barriers

View the barrier assets from our combined attribute + text search.

In [ ]:
uva.viewer.object_show(barrier_uids)

## 7. Material Assets

UrbanVerse-100K includes PBR materials for sky (646 HDRIs), road (98), sidewalk (190),
and terrain (115). Each module shares the same API pattern:
`get_descriptions()` → `get_descriptions_conditioned()` → `load_materials()` → viewer.

### 7.1 Sky Materials

In [ ]:
sunny = uva.sky.get_descriptions_conditioned(query="Sunny summer day", top_k=10)
print(f"Top sky matches: {sunny}")

uva.sky.load_materials(sunny)
uva.viewer.sky_show(sunny)

### 7.2 Road Materials

In [ ]:
highway = uva.road.get_descriptions_conditioned(query="Highway", top_k=10)
print(f"Top road matches: {highway}")

uva.road.load_materials(highway)
uva.viewer.road_show(highway)

### 7.3 Sidewalk Materials

In [ ]:
stone = uva.sidewalk.get_descriptions_conditioned(query="Bumpy stone road", top_k=10)
print(f"Top sidewalk matches: {stone}")

uva.sidewalk.load_materials(stone)
uva.viewer.sidewalk_show(stone)

### 7.4 Terrain Materials

In [ ]:
grass = uva.terrain.get_descriptions_conditioned(query="Grassland", top_k=10)
print(f"Top terrain matches: {grass}")

uva.terrain.load_materials(grass)
uva.viewer.terrain_show(grass)

## 8. Vegetation Assets

Vegetation modules (`uva.plant`, `uva.shrub`, `uva.tree`) provide USD-format
assets for plants, shrubs, and trees. They follow the same API pattern as materials.

**Note:** These assets are redistributed from the official IsaacSim asset library, as we found that IsaacSim does not render other tree-like assets reliably.

In [ ]:
trees = uva.tree.get_descriptions_conditioned(query="tall oak tree", top_k=10)
print(f"Top tree matches: {trees}")

uva.tree.load_materials(trees)
uva.viewer.tree_show(trees)

In [ ]:
plants = uva.plant.get_descriptions()
print(f"Available plants: {len(plants)}")

uva.plant.load_materials(plants[:5])
uva.viewer.plant_show(plants[:5])

To download the **complete dataset**—including all 3D objects (`.glb`, `.json` annotations, `.png` thumbnails, `.png` renders), as well as all materials (sky, road, sidewalk, terrain) and vegetation assets (trees, shrubs, plants), we can use the following command:

In [ ]:
# WARNING: This downloads the full dataset (~1.18 TB).
uva.download_all(num_workers=32)

> **Note:** If the download is interrupted for any reason, you can simply rerun the function above to resume the process, or use the repair function below. Both methods allow the download to reliably continue from the point where it was interrupted.

## 10. Integrity Check & Repair

After downloading, you can verify that all files are present and intact, and automatically repair any missing ones.

In [ ]:
# Output a report of the integrity of the downloaded files, including the number of complete and missing files.
report = uva.check_integrity()
print(f"Complete: {report['complete']}")
print(f"Missing or incomplete files: {report['total_missing']}")

In [ ]:
# Automatically download only the missing or incomplete files
report = uva.repair(num_workers=16)